In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
import copy
import collections # Import collections for Counter

# --- Configuration for isolation ---
data_dir = '/content/drive/MyDrive/DAiSEE_Processed_Faces_Binary'

# Check if DAiSEE dataset directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"DAiSEE dataset not found at {data_dir}. "
                            "Please ensure the dataset is unzipped correctly and copied to Drive.")

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset
import os
import copy
import collections # Import collections for Counter
from PIL import Image
import re

print("--- Vision Transformer (ViT) Model Setup (Isolated) ---")

# --- Configuration for isolation ---
data_dir = '/content/drive/MyDrive/DAiSEE_Processed_Faces_Binary'

# Check if DAiSEE dataset directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"DAiSEE dataset not found at {data_dir}. "
                            "Please ensure the dataset is unzipped correctly and copied to Drive.")

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# --- SingleFrameImageDataset and get_one_frame_per_video_samples ---
class SingleFrameImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

def get_one_frame_per_video_samples(image_folder_dataset):
    filtered_samples = []
    video_ids_processed = set()

    for path, class_idx in image_folder_dataset.samples:
        filename = os.path.basename(path)
        match = re.match(r'(.+)_face_\d{5}\.jpg', filename)
        if match:
            video_id = match.group(1)
        else:
            video_id = filename.split('.')[0]

        if video_id not in video_ids_processed:
            filtered_samples.append((path, class_idx))
            video_ids_processed.add(video_id)
    return filtered_samples


# --- Data Transformations ---
data_transforms = {
    'Train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'Validation': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# --- Dataset and DataLoader ---
full_image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['Train', 'Validation']}

filtered_samples = {
    'Train': get_one_frame_per_video_samples(full_image_datasets['Train']),
    'Validation': get_one_frame_per_video_samples(full_image_datasets['Validation'])
}

image_datasets = {
    'Train': SingleFrameImageDataset(filtered_samples['Train'], data_transforms['Train']),
    'Validation': SingleFrameImageDataset(filtered_samples['Validation'], data_transforms['Validation'])
}

dataloaders = {x: DataLoader(image_datasets[x], batch_size=32,
                                            shuffle=True, num_workers=2)
               for x in ['Train', 'Validation']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['Train', 'Validation']}
class_names = ['Not Engaged', 'Engaged']
num_classes = len(class_names)
print(f"Detected classes: {class_names}")
print(f"Number of classes: {num_classes}")

# --- Training Function ---
def train_model(model, dataloaders, criterion, optimizer, num_epochs=20):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['Train', 'Validation']:
            if phase == 'Train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            for batch_idx, (inputs, labels) in enumerate(dataloaders[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'Train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'Train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                if batch_idx % 100 == 0:
                    print(f'  {phase} Batch {batch_idx}/{len(dataloaders[phase])} Loss: {running_loss / ((batch_idx + 1) * dataloaders[phase].batch_size):.4f} Acc: {running_corrects.double() / ((batch_idx + 1) * dataloaders[phase].batch_size):.4f}')

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'Validation' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

print("train_model function defined for ViT section.")


# --- Vision Transformer (ViT) Model Setup ---

# Load a pre-trained Vision Transformer model
vit_model = models.vit_b_16(weights='IMAGENET1K_V1')

# Modify the classifier head for binary classification
# The Vision Transformer has a `heads` attribute, which contains the `head` layer
num_ftrs_vit = vit_model.heads.head.in_features
vit_model.heads.head = nn.Linear(num_ftrs_vit, num_classes)

# Move model to the selected device
vit_model = vit_model.to(device);

# --- Loss Function and Optimizer for ViT ---
criterion_vit = nn.CrossEntropyLoss()
optimizer_vit = optim.SGD(vit_model.parameters())

print("Starting ViT model training...")
vit_model = train_model(vit_model, dataloaders, criterion_vit, optimizer_vit, num_epochs=20)

# --- Save the trained ViT model ---
vit_model_save_path = 'best_finetuned_vit_model.pt'
torch.save(vit_model.state_dict(), vit_model_save_path)
print(f"ViT model saved to {vit_model_save_path}")

--- Vision Transformer (ViT) Model Setup (Isolated) ---
Using device: cuda
Detected classes: ['Not Engaged', 'Engaged']
Number of classes: 2
train_model function defined for ViT section.
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 203MB/s]


Starting ViT model training...
Epoch 0/19
----------
  Train Batch 0/163 Loss: 0.7204 Acc: 0.4688
  Train Batch 100/163 Loss: 0.6092 Acc: 0.6912
Train Loss: 0.5991 Acc: 0.6994
  Validation Batch 0/42 Loss: 0.7074 Acc: 0.5938
Validation Loss: 0.7138 Acc: 0.5618

Epoch 1/19
----------
  Train Batch 0/163 Loss: 0.6258 Acc: 0.6562
  Train Batch 100/163 Loss: 0.5778 Acc: 0.7135
Train Loss: 0.5676 Acc: 0.7240
  Validation Batch 0/42 Loss: 0.6153 Acc: 0.6875
Validation Loss: 0.6961 Acc: 0.5924

Epoch 2/19
----------
  Train Batch 0/163 Loss: 0.5122 Acc: 0.7188
  Train Batch 100/163 Loss: 0.5530 Acc: 0.7355
Train Loss: 0.5570 Acc: 0.7329
  Validation Batch 0/42 Loss: 0.7270 Acc: 0.5938
Validation Loss: 0.6724 Acc: 0.6118

Epoch 3/19
----------
  Train Batch 0/163 Loss: 0.5147 Acc: 0.7500
  Train Batch 100/163 Loss: 0.5486 Acc: 0.7336
Train Loss: 0.5525 Acc: 0.7323
  Validation Batch 0/42 Loss: 0.7074 Acc: 0.5625
Validation Loss: 0.7167 Acc: 0.5991

Epoch 4/19
----------
  Train Batch 0/163 Los

In [5]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# --- Configuration (reuse from training cell) ---
data_dir = '/content/drive/MyDrive/DAiSEE_Processed_Faces_Binary'
vit_model_save_path = 'best_finetuned_vit_model.pt' # Path to the saved ViT model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Data Transformations for Test Set ---
# Use the same transformations as validation set
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Load Test Dataset and DataLoader ---
# Original ImageFolder dataset
full_test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'Test'), test_transforms)

# Filter samples to get one frame per video (using the helper from earlier cells)
filtered_test_samples_vit = get_one_frame_per_video_samples(full_test_dataset)

# Create new custom test dataset with filtered samples
test_dataset_vit = SingleFrameImageDataset(filtered_test_samples_vit, test_transforms)

test_dataloader_vit = DataLoader(test_dataset_vit, batch_size=32, shuffle=False, num_workers=2)
test_dataset_size_vit = len(test_dataset_vit)

# Explicitly set test class names for clarity in binary classification
test_class_names = ['Not Engaged', 'Engaged']
num_classes = len(test_class_names)

print(f"Detected test classes: {test_class_names}")
print(f"Number of test samples for ViT: {test_dataset_size_vit}")

# --- Model Setup for Evaluation ---
# Load the pre-trained ViT model architecture
vit_model_eval = models.vit_b_16(weights='IMAGENET1K_V1')
num_ftrs_vit = vit_model_eval.heads.head.in_features
vit_model_eval.heads.head = nn.Linear(num_ftrs_vit, num_classes)

# Load the best weights from the saved ViT model
if os.path.exists(vit_model_save_path):
    vit_model_eval.load_state_dict(torch.load(vit_model_save_path, map_location=device))
    print(f"Successfully loaded ViT model weights from {vit_model_save_path}")
else:
    raise FileNotFoundError(f"Saved ViT model not found at {vit_model_save_path}. "
                            "Please ensure the ViT training cell was run and model saved.")

vit_model_eval = vit_model_eval.to(device);
vit_model_eval.eval() # Set model to evaluation mode

# --- Evaluation Function (re-defining for clarity/isolation) ---
def evaluate_model(model, dataloader, criterion):
    running_loss = 0.0
    running_corrects = 0
    all_preds = []
    all_labels = []

    # Iterate over data.
    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        with torch.no_grad(): # No need to track gradients for evaluation
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    total_loss = running_loss / len(dataloader.dataset)
    total_acc = running_corrects.double() / len(dataloader.dataset)

    print(f'ViT Test Loss: {total_loss:.4f} Acc: {total_acc:.4f}')
    return total_loss, total_acc, all_preds, all_labels

# --- Start Evaluation ---
print("Starting ViT model evaluation on test set...")
criterion_vit_eval = nn.CrossEntropyLoss() # Use the same criterion as during training
vit_test_loss, vit_test_accuracy, all_preds_vit, all_labels_vit = evaluate_model(vit_model_eval, test_dataloader_vit, criterion_vit_eval)
print(f"Final ViT Test Accuracy: {vit_test_accuracy:.4f}")

Detected test classes: ['Not Engaged', 'Engaged']
Number of test samples for ViT: 1713
Successfully loaded ViT model weights from best_finetuned_vit_model.pt
Starting ViT model evaluation on test set...
ViT Test Loss: 0.5862 Acc: 0.7157
Final ViT Test Accuracy: 0.7157


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating detailed performance metrics for ViT...")

# Convert to numpy arrays
all_preds_vit = np.array(all_preds_vit)
all_labels_vit = np.array(all_labels_vit)

# Define binary class names for reporting
binary_class_names = ['Not Engaged', 'Engaged']

# 1. Classification Report
print("\nViT Classification Report:")
print(classification_report(all_labels_vit, all_preds_vit, target_names=binary_class_names))

# 2. Confusion Matrix
cm_vit = confusion_matrix(all_labels_vit, all_preds_vit)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_vit, annot=True, fmt="d", cmap="Blues",
            xticklabels=binary_class_names, yticklabels=binary_class_names)
plt.title("ViT Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
import torch.nn.functional as F

print("Calculating Precision-Recall Curve for ViT...")

all_probs_vit_pr = []
all_labels_vit_pr = []

vit_model_eval.eval() # Ensure model is in evaluation mode
with torch.no_grad():
    for inputs, labels in test_dataloader_vit:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = vit_model_eval(inputs)
        # Get probabilities for the positive class (Engaged, which is index 1)
        probs = F.softmax(outputs, dim=1)[:, 1]
        all_probs_vit_pr.extend(probs.cpu().numpy())
        all_labels_vit_pr.extend(labels.cpu().numpy())

# Calculate precision and recall
precision_vit, recall_vit, _ = precision_recall_curve(all_labels_vit_pr, all_probs_vit_pr)
auc_score_vit = auc(recall_vit, precision_vit)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall_vit, precision_vit, label=f'ViT (AUC = {auc_score_vit:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for ViT')
plt.legend()
plt.grid(True)
plt.show()

print(f"ViT Precision-Recall AUC: {auc_score_vit:.2f}")

In [ ]:
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns

# Define the target directory in Google Drive for results
drive_results_dir_vit = os.path.join('/content/drive/MyDrive/DAiSEE_Processed_Faces_Binary', 'results_vit')
os.makedirs(drive_results_dir_vit, exist_ok=True)

# 1. Save the trained ViT model to Google Drive
saved_vit_model_filename = 'best_finetuned_vit_model.pt'
source_vit_model_path = saved_vit_model_filename
target_vit_model_path = os.path.join(drive_results_dir_vit, saved_vit_model_filename)

if os.path.exists(source_vit_model_path):
    shutil.copy(source_vit_model_path, target_vit_model_path)
    print(f"Trained ViT model '{saved_vit_model_filename}' successfully copied to Drive: {target_vit_model_path}")
else:
    print(f"Warning: Trained ViT model '{saved_vit_model_filename}' not found at {source_vit_model_path}. Skipping save to Drive.")

# 2. Save the ViT Confusion Matrix plot to Google Drive
# Re-generate the confusion matrix plot to save it
plt.figure(figsize=(8, 6))
sns.heatmap(cm_vit, annot=True, fmt="d", cmap="Blues",
            xticklabels=binary_class_names, yticklabels=binary_class_names)
plt.title("ViT Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

confusion_matrix_save_path_vit = os.path.join(drive_results_dir_vit, 'confusion_matrix_vit.png')
plt.savefig(confusion_matrix_save_path_vit)
plt.close() # Close the plot to prevent it from being displayed twice
print(f"ViT Confusion Matrix plot saved to Drive: {confusion_matrix_save_path_vit}")

# 3. Save the ViT Precision-Recall Curve plot to Google Drive
# Re-generate the PR curve plot to save it
plt.figure(figsize=(8, 6))
plt.plot(recall_vit, precision_vit, label=f'ViT (AUC = {auc_score_vit:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for ViT')
plt.legend()
plt.grid(True)

pr_curve_save_path_vit = os.path.join(drive_results_dir_vit, 'precision_recall_curve_vit.png')
plt.savefig(pr_curve_save_path_vit)
plt.close() # Close the plot
print(f"ViT Precision-Recall Curve plot saved to Drive: {pr_curve_save_path_vit}")